<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/sleep/04_varying_intercept.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sleep deprivation 4 — Varying intercepts

In the previous notebooks, every participant shared the same regression line.

Here we build our first hierarchical model. Participants will have different baseline reaction times while sharing one population-level effect of sleep deprivation.

## Setup

This course pins PyMC and the modular ArviZ packages for reproducibility because their APIs can change across major versions.

In [ ]:
%pip install -q \
    "pandas==2.2.3" \
    "pymc==6.3.2" \
    "arviz-base==1.3.0" \
    "arviz-stats==1.3.2" \
    "arviz-plots[matplotlib]==1.3.1"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import pymc as pm
import arviz_base as azb
import arviz_plots as azp
import arviz_stats as azs

RANDOM_SEED = 20260924
azp.style.use("arviz-variat")

print("PyMC:", pm.__version__)
print("arviz-base:", azb.__version__)
print("arviz-plots:", azp.__version__)
print("arviz-stats:", azs.__version__)

## Data

The original study contains two adaptation/training days followed by a baseline measurement and then seven nights of severe sleep restriction. You can read about the original study here: [Belenky J Sleep Res. 2003](https://doi.org/10.1046/j.1365-2869.2003.00337.x)

Following the chapter, we drop original days 0–1 and subtract 2 from the remaining day number. Therefore **`Days = 0` is the baseline measurement before sleep deprivation begins**.

That zero point is scientifically meaningful, so every model in this sequence keeps `Days` on its natural scale.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/lme4/sleepstudy.csv"

sleep = pd.read_csv(DATA_URL).drop(columns="rownames")
sleep = sleep.loc[sleep["Days"] >= 2].copy()
sleep["Days"] = sleep["Days"] - 2
sleep["Subject"] = sleep["Subject"].astype(str)
sleep = sleep.reset_index(drop=True)

print(f"{sleep['Subject'].nunique()} participants, {len(sleep)} observations")
print(f"Days: {sleep['Days'].min()} to {sleep['Days'].max()}")
sleep.head()

### A different look at the data

Instead of plotting all of the trajectories again, compare each participant's reaction time at baseline with the same participant's reaction time on day 7.

In [ ]:
rt_by_day = sleep.pivot(index="Subject", columns="Days", values="Reaction")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(rt_by_day[0], rt_by_day[7])
ax.set(
    xlabel="Reaction time at baseline, day 0 (ms)",
    ylabel="Reaction time at day 7 (ms)",
)
plt.show()

Participants who are relatively fast at baseline also tend to be relatively fast later in the experiment. We want a model that can preserve these participant differences while still estimating an effect of sleep deprivation.

### Plotting helper

The participant plotting helper now accepts an optional `coords` argument. When supplied, it passes those coordinate selections directly to `azp.plot_lm`.

In [ ]:
LM_VISUALS = {
    "pe_line": {"color": "C1"},
    "ci_band": {"color": "C0"},
    "observed_scatter": {"color": "black", "alpha": 1, "zorder": 3, "s": 10},
}

PARTICIPANT_DAY = xr.Coordinates.from_pandas_multiindex(
    pd.MultiIndex.from_frame(
        sleep[["Subject", "Days"]],
        names=["participant", "day"],
    ),
    "obs_id",
)

def plot_participants(dt, group, var, coords=None):
    """One panel per participant, optionally restricted with ArviZ coords."""
    def reshape(ds):
        return ds.assign_coords(PARTICIPANT_DAY).unstack("obs_id")

    panels = xr.DataTree.from_dict({
        group: reshape(dt[group].to_dataset()),
        "observed_data": reshape(dt["observed_data"].to_dataset()),
        "constant_data": reshape(dt["constant_data"].to_dataset()),
    })

    pc = azp.plot_lm(
        panels,
        x="days",
        y=var,
        y_obs="y",
        group=group,
        plot_dim="day",
        coords=coords,
        ci_prob=(0.50, 0.90),
        ci_kind="hdi",
        point_estimate="mean",
        smooth=False,
        cols=["participant"],
        col_wrap=6,
        figure_kwargs={"figsize": (11, 5.5), "sharex": True, "sharey": True},
        visuals={**LM_VISUALS, "xlabel": False, "ylabel": False},
    )

    pc.add_legend("prob", title="HDI")

    fig = pc.get_viz("figure")
    fig.supxlabel("Days of sleep deprivation")
    fig.supylabel("Reaction time (ms)")

    return pc

## 1. Build a varying-intercept model

### 1.1 What model can preserve participant differences and still estimate a sleep effect?

What model can express the idea that people with faster reaction times at the beginning generally have faster reaction times at the end and also still have a parameter for how sleep affects reaction time?

We will give every participant their own intercept while keeping one shared slope:

$$
y_i \sim \operatorname{Normal}(\mu_{y,i}, sd_y)
$$

$$
\mu_{y,i} = b_{0,s[i]} + b_1\,\mathrm{days}_i
$$

$$
b_{0,s} \sim \operatorname{Normal}(\mu_{b0}, sd_{b0})
$$

Here `s[i]` identifies the participant who produced observation `i`.

### 1.2 Which regression parameter varies across participants?

Which regression parameter has one value for each participant, and which regression parameter is shared by everyone?

- answer here

### 1.3 Create the participant labels and observation-to-participant index.

We need an ordered list of participant labels and, for every observation, the integer position of that participant in the list.

In [ ]:
# answer here

### 1.4 We will build the model incrementally.

A PyMC model does not have to be written in one `with pm.Model():` block. We can create the model first and then add nodes in later `with model:` blocks.

This lets us inspect the model as it grows.

### 1.5 What coordinate values belong to the observation dimension?

The coordinate named `obs_id` needs one label for each row of the dataset. What values should it contain?

- answer here

### 1.6 What coordinate values belong to the participant dimension?

The coordinate named `participant` needs one label for each participant. What values should it contain?

- answer here

### 1.7 Create the coordinate dictionary.

In [ ]:
# answer here

### 1.8 Create the empty model.

In [ ]:
# answer here

### 1.9 What dimension does `days` use?

There is one value of `days` for each observation. Which model dimension should it use?

- answer here

### 1.10 Add `days` to the model.

In [ ]:
# answer here

### 1.11 What dimension does `participant_idx` use?

There is one participant index for every observation. Which model dimension should it use?

- answer here

### 1.12 Add `participant_idx` to the model.

In [ ]:
# answer here

### 1.13 What does `mu_b0` represent?

What quantity is represented by `mu_b0` in the hierarchical model?

- answer here

### 1.14 Choose the hyperprior parameters for `mu_b0`.

As in Notebook 1, we are fairly sure that almost all population-average baseline reaction times lie between 50 and 450 ms.

What Normal distribution puts approximately 95% of its probability between those limits?

- answer here

### 1.15 Store the hyperprior parameters for `mu_b0`.

In [ ]:
# answer here

### 1.16 What distribution does `mu_b0` have?

Write the prior for `mu_b0` using the constants from the previous cell.

- answer here

### 1.17 Add `mu_b0` to the model.

In [ ]:
# answer here

### 1.18 What does `sd_b0` represent?

What quantity is represented by `sd_b0` in the hierarchical model?

- answer here

### 1.19 Choose the hyperprior parameter for `sd_b0`.

Suppose we use an Exponential hyperprior and want its scale to be 25 ms.

What value should be stored in `sd_sd_b0`?

- answer here

### 1.20 Store the hyperprior parameter for `sd_b0`.

In [ ]:
# answer here

### 1.21 What distribution does `sd_b0` have?

Write the prior for `sd_b0` using `sd_sd_b0`.

- answer here

### 1.22 Add `sd_b0` to the model.

In [ ]:
# answer here

### 1.23 What distribution does each participant-specific `b0` have?

Use the population parameters already in the model.

- answer here

### 1.24 What dimension does `b0` use?

There is one `b0` value for each participant. Which dimension should be assigned to it?

- answer here

### 1.25 Add the participant-specific `b0` values to the model.

In [ ]:
# answer here

### 1.26 What is the size of `b1`?

Does this model contain one `b1` for each participant or one `b1` shared by the population?

- answer here

### 1.27 What prior parameters did we use for `b1` in Notebook 1?

The broad slope prior placed approximately 95% of its probability between −40 and +40 ms/day. What values should `mu_b1` and `sd_b1` have?

- answer here

### 1.28 Store the prior parameters for `b1`.

In [ ]:
# answer here

### 1.29 What distribution does `b1` have?

- answer here

### 1.30 Add `b1` to the model.

In [ ]:
# answer here

### 1.31 What is the size of `sd_y`?

Does the model contain one likelihood scale per participant or one likelihood scale shared by all observations?

- answer here

### 1.32 What prior mean did we use for `sd_y` in Notebook 1?

The Exponential prior for `sd_y` had mean 50 ms. What value should be stored in `mu_sd_y`?

- answer here

### 1.33 Store the prior parameter for `sd_y`.

In [ ]:
# answer here

### 1.34 What distribution does `sd_y` have?

- answer here

### 1.35 Add `sd_y` to the model.

In [ ]:
# answer here

### 1.36 How do we select the correct intercept for each observation?

`b0` is indexed by participant. What PyTensor indexing expression returns the participant-specific intercept corresponding to every row of the data?

- answer here

### 1.37 Construct the per-observation intercept expression.

In [ ]:
# answer here

### 1.38 What equation gives the expected reaction time `mu_y`?

Use the per-observation intercept, the common slope, and `days`.

- answer here

### 1.39 Construct the expression for `mu_y`.

In [ ]:
# answer here

### 1.40 What dimension does `mu_y` use?

There is one expected reaction time for every observation. Which dimension should `mu_y` use?

- answer here

### 1.41 Add `mu_y` to the model.

In [ ]:
# answer here

### 1.42 What distribution does the observed outcome `y` have?

Write the likelihood using the quantities already in the model.

- answer here

### 1.43 What dimension does `y` use?

There is one observed reaction time for every observation. Which dimension should `y` use?

- answer here

### 1.44 Add `y` and complete the model.

In [ ]:
# answer here

## 2. Examine the model

### 2.1 What Python command gives a textual representation of the model?

- answer here

### 2.2 Print the completed model.

In [ ]:
# answer here

### 2.3 What PyMC function draws a graphical representation of the model?

- answer here

### 2.4 Draw the model graph.

In [ ]:
# answer here

## 3. Prior predictive check

### 3.1 What criteria will we use?

Use the prior-predictive criteria established in Notebook 1:

1. predicted reaction times should not routinely be physically impossible;
2. baseline reaction times should mostly occupy a broadly plausible range;
3. the model should allow plausible changes across the experiment without routinely generating absurd trajectories.

The hierarchical model adds one new criterion:

4. it should allow meaningful differences among participant baselines without routinely producing implausibly large between-participant differences.

### 3.2 Draw from the prior and prior predictive distributions.

Draw 500 samples and retain `mu_b0`, `sd_b0`, `b0`, `b1`, `sd_y`, `mu_y`, and `y`.

In [ ]:
# answer here

### 3.3 Which variables describe the population-level prior?

Which four scalar variables describe the population-level baseline distribution, common slope, and likelihood scale?

- answer here

### 3.4 Plot the population-level priors.

Use `azp.plot_dist` with `group="prior"` for `mu_b0`, `sd_b0`, `b1`, and `sd_y`.

In [ ]:
# answer here

### 3.5 What should these population-level priors allow?

Use these concrete criteria:

- `mu_b0`: population-average baseline reaction times should mostly be in the broad 50–450 ms range;
- `sd_b0`: participant baselines should be allowed to differ substantially, but extremely large between-participant spreads should not be routine;
- `b1`: retain the broad slope prior from Notebook 1;
- `sd_y`: retain the broad positive residual-scale prior from Notebook 1.

### 3.6 Do the population-level priors satisfy those criteria?

- answer here

### 3.7 Plot the prior predictive reaction times.

Use `plot_participants` to plot prior predictive `y`.

In [ ]:
# answer here

### 3.8 Do the prior predictive samples satisfy the criteria from Question 3.1?

- answer here

## 4. Fit the hierarchical model

### 4.1 Sample from the posterior.

Draw 1000 posterior samples in each of 4 chains after 1500 tuning samples.

In [ ]:
# answer here

## 5. Diagnose the fit

### 5.1 Which parameters should we check at the population level?

Which scalar posterior quantities describe the population baseline distribution, shared slope, and likelihood scale?

- answer here

### 5.2 Calculate the numerical diagnostics for the population-level parameters.

Count divergences and use `azs.summary` with a 90% HDI.

In [ ]:
# answer here

### 5.3 Plot the sampling diagnostics for the population-level parameters.

Use `azp.plot_trace_dist`.

In [ ]:
# answer here

### 5.4 Do the population-level parameters meet the diagnostic criteria from Notebook 1?

- answer here

### 5.5 Why not plot diagnostics for all participant-specific intercepts at once?

`b0` contains one posterior parameter for every participant. A smaller representative subset is easier to inspect.

### 5.6 Which ArviZ argument selects particular coordinate values?

Which argument can be used in both `azs.summary` and `azp.plot_trace_dist` to select specific participants?

- answer here

### 5.7 Choose a small representative subset of participants.

In [ ]:
# answer here

### 5.8 Calculate diagnostics for the selected participant-specific `b0` values.

In [ ]:
# answer here

### 5.9 Plot diagnostics for the same `b0` values.

In [ ]:
# answer here

### 5.10 Do the inspected participant-level parameters meet the diagnostic criteria?

- answer here

## 6. Examine the fitted hierarchy

### 6.1 What does `mu_b0` represent after fitting?

What posterior quantity gives the population-average baseline reaction time?

- answer here

### 6.2 Plot the posterior distribution of `mu_b0`.

In [ ]:
# answer here

### 6.3 What does `sd_b0` represent after fitting?

Which posterior quantity measures between-participant variation in expected baseline reaction time?

- answer here

### 6.4 Plot the posterior distribution of `sd_b0`.

In [ ]:
# answer here

### 6.5 What does `sd_y` represent after fitting?

Which posterior quantity measures observation-to-observation variation around `mu_y`?

- answer here

### 6.6 Plot the posterior distribution of `sd_y`.

In [ ]:
# answer here

### 6.7 Which ArviZ plot is designed to show intervals for many one-dimensional posterior quantities compactly?

We want to display the participant-specific `b0` values without making 18 separate density plots.

- answer here

### 6.8 Plot the participant-specific posterior intercepts.

Use `azp.plot_forest` for `b0`, combining chains and showing 50% and 90% intervals.

In [ ]:
# answer here

### 6.9 What is partial pooling?

Because all participant-specific `b0` values are estimated through the shared population distribution `Normal(mu_b0, sd_b0)`, information is shared across participants rather than each participant being fitted independently.

This is **partial pooling**.

### 6.10 What does `b1` represent in this model?

Which posterior quantity is the common change in expected reaction time for one additional day of sleep deprivation?

- answer here

### 6.11 Plot the posterior distribution of `b1`.

In [ ]:
# answer here

### 6.12 Compare uncertainty in `b1` with Notebook 1.

Notebook 1 gave a 90% HDI for the population slope of approximately 7.94 to 14.07 ms/day.

Calculate the width of that interval and the width of the 90% HDI for `b1` in the current model.

In [ ]:
# answer here

### 6.13 Which model gives the more precise estimate of the common slope?

Use the calculated HDI widths.

- answer here

## 7. Posterior predictive check

### 7.1 What criteria will we use?

Use the established posterior-predictive criteria:

1. overall reaction-time level;
2. change across days;
3. observation-to-observation variation.

For this notebook, the main new question is whether the model reproduces persistent differences in overall reaction-time level among participants.

### 7.2 Generate posterior predictive reaction times.

In [ ]:
# answer here

### 7.3 Which `plot_lm` argument selects particular participant coordinate values?

The supplied `plot_participants` helper passes this argument through to `azp.plot_lm`.

- answer here

### 7.4 Plot posterior predictive reaction times.

First plot all participants.

In [ ]:
# answer here

### 7.5 Does the model reproduce persistent participant differences in overall reaction-time level?

Apply the criterion stated in Question 7.1.

- answer here

## 8. What limitation remains?

### 8.1 How many slope parameters are in the model?

Does the model contain one `b1` for every participant or one `b1` shared by all participants?

- answer here

### 8.2 What does that imply about participant trajectories?

If participants have different `b0` values but all share the same `b1`, what must be true about their expected regression lines?

- answer here

### 8.3 Plot a subset of participant-specific expected trajectories.

Use the same `coords` mechanism to make the parallel slopes easy to see.

In [ ]:
# answer here